In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # to read the data
import kagglehub
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import f1_score

from sklearn.model_selection import StratifiedKFold


import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

import os # to read the data


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
data_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean["Target"].isnull().sum()

In [ ]:
# Task 1: Write your code here:
null_cols = list(df_clean.drop("Target", axis=1).isnull())

# i will fill the missing values with the median
# from what i understand the each feature is a code for something (categorical) so for that i think the median makes more sense ;

df_clean[null_cols] = df_clean[null_cols].fillna(df_clean[null_cols].median())

In [ ]:
# Task 2: Write your code here:
df_clean.duplicated().sum().any()
# there aren't any duplicated rows

In [ ]:
# Task 3: Write your code here:
df_clean.select_dtypes(include='object').columns

# there are not any categorical columns

In [ ]:
# Task 4: Write your code here:
feature_cols = df_clean.columns.drop("Target")
feature_cols

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

In [ ]:
# Task 5: Write your code here:

df['Target'].value_counts(normalize=True)

# the target is imbalanced

In [ ]:
# Task 1: Write your code here:
X = df_clean[feature_cols]
y = df_clean['Target']

In [ ]:
%pip install catboost -q
from catboost import CatBoostClassifier

In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(verbose=0, n_estimators=320, max_depth=4)
all_f1 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]



  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # f1 : because the data is imbalanced
  f1 = f1_score(y_test, y_pred)
  all_f1.append(f1)



print(f"avg f1: {np.mean(all_f1)}")

In [ ]:

# Task 1: Write your code here:
importance = model.feature_importances_
sorted_imp = np.argsort(importance)


feature_cols = feature_cols[sorted_imp]
importance = importance[sorted_imp]


plt.figure(figsize=(10, 50))
plt.barh(feature_cols, importance)
plt.title(f"Random Forst Importance")
plt.xlabel("Coefficient  (Importance)")
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
# from the feature importance plot above
print("The most important feature is D_136")


In [ ]:
# Task Bonus: Write your code here: